# 18.8 AI 治理与合规 (EU AI Act & Governance)

> 🕐 预估学习时间：35分钟

EU AI Act、GDPR、行业规范要求把“模型上线”变成可审计的治理流程：风险分级、数据谱系、透明披露、人工监督与事件响应。本节用可执行的检查清单与证据包结构落地。

本节涵盖：
- 风险分级（不可接受 / 高 / 有限 / 最小）
- 高风险义务映射到工程控件
- 模型卡 / 数据卡 / 评测证据包
- 运行时治理（日志、人工复核、撤回）
- 合规门禁自动化


## 1. 风险分级速查

| 级别 | 例子 | 义务强度 |
|------|------|---------|
| 不可接受 | 社会评分、裸露生物识别滥用 | 禁止 |
| 高风险 | 雇佣、信贷、关键基础设施、教育录取 | 完整合规 |
| 有限风险 | 聊天机器人 | 透明披露 |
| 最小风险 | 多数推荐/滤镜 | 自愿准则 |

通用 GPAI / 系统性风险模型另有训练数据摘要、版权、评测与事件报告义务。


In [ ]:
from dataclasses import dataclass, field


@dataclass
class SystemProfile:
    name: str
    domain: str
    uses_biometrics: bool = False
    decides_employment_or_credit: bool = False
    is_chatbot: bool = False
    open_ended_general: bool = False
    users_eu: bool = True


def classify_risk(p: SystemProfile) -> str:
    if p.uses_biometrics and p.domain in {'surveillance', 'social_scoring'}:
        return 'unacceptable'
    if p.decides_employment_or_credit or p.domain in {'critical_infra', 'education_admission'}:
        return 'high'
    if p.is_chatbot:
        return 'limited'
    if p.open_ended_general:
        return 'gpai'
    return 'minimal'


profiles = [
    SystemProfile('social-score', 'social_scoring', uses_biometrics=True),
    SystemProfile('hire-bot', 'hr', decides_employment_or_credit=True),
    SystemProfile('support-bot', 'saas', is_chatbot=True),
    SystemProfile('gpai-base', 'foundation', open_ended_general=True),
]
print('=== Risk Classification ===')
for p in profiles:
    print(f'{p.name}: {classify_risk(p)}')
print(f'\nKey: Classification drives which engineering controls are mandatory vs optional.')


## 2. 高风险义务 → 工程控件映射

| 法律义务 | 工程落地 |
|---------|---------|
| 风险管理 | 威胁建模 + 残留风险登记 |
| 数据治理 | 来源、许可、PII、偏见切片评估 |
| 技术文档 | 模型卡、架构图、训练配置冻结 |
| 记录保存 | 不可篡改推理日志 / 版本指纹 |
| 透明性 | 用户告知 AI 交互、能力边界 |
| 人工监督 | 高影响动作人工确认 |
| 准确性/鲁棒/安全 | 评测门禁 + 红队 + 护栏 |


In [ ]:
CONTROLS = {
    'risk_register': False,
    'data_lineage': False,
    'model_card': False,
    'eval_gate': False,
    'immutable_logs': False,
    'user_disclosure': False,
    'human_in_loop': False,
    'red_team': False,
    'incident_runbook': False,
}


def required_controls(risk: str) -> list[str]:
    base = ['user_disclosure']
    if risk in {'limited'}:
        return base
    if risk in {'gpai'}:
        return base + ['model_card', 'data_lineage', 'eval_gate', 'immutable_logs', 'incident_runbook']
    if risk == 'high':
        return list(CONTROLS)
    return []


def compliance_gap(risk: str, enabled: dict) -> dict:
    need = required_controls(risk)
    missing = [c for c in need if not enabled.get(c, False)]
    return {'required': need, 'missing': missing, 'ready': len(missing) == 0}


enabled = dict(CONTROLS)
enabled.update(user_disclosure=True, model_card=True, eval_gate=True)
print('=== Control Gap (high-risk hire-bot) ===')
print(compliance_gap('high', enabled))
print(f'\nKey: Translate legal text into a boolean control plane checked in CI/CD.')


## 3. 证据包：模型卡 + 数据卡 + 评测

发布前冻结一组可审计产物，哈希入库。


In [ ]:
import hashlib
import json


def evidence_pack(model_name, metrics, data_sources, risks):
    pack = {
        'model': model_name,
        'model_card': {
            'intended_use': 'customer support drafting',
            'out_of_scope': ['legal advice', 'medical diagnosis'],
            'metrics': metrics,
        },
        'data_card': {
            'sources': data_sources,
            'pii_scrubbed': True,
            'licenses': ['MIT', 'CC-BY-4.0'],
        },
        'risks': risks,
    }
    blob = json.dumps(pack, sort_keys=True).encode()
    pack['sha256'] = hashlib.sha256(blob).hexdigest()
    return pack


pack = evidence_pack(
    'support-llm-v3',
    {'mt_bench': 7.8, 'toxicity_rate': 0.012},
    ['curated_tickets', 'public_docs'],
    ['prompt_injection', 'hallucination'],
)
print('=== Evidence Pack ===')
print(json.dumps(pack, indent=2)[:500], '...')
print(f'\nKey: Hashed evidence packs make releases auditable and rollback-friendly.')


## 4. 运行时治理与事件响应

- 高风险动作（转账建议、解雇建议）必须 HITL  
- 日志含：模型版本、策略版本、输入哈希、护栏决策  
- 事件：检测 → 遏制（关特性）→ 通知 → 根因 → 回归测试


In [ ]:
class RuntimeGovernor:
    def __init__(self, hitl_actions=None):
        self.hitl_actions = set(hitl_actions or [])
        self.events = []

    def authorize(self, action, user_approved=False):
        if action in self.hitl_actions and not user_approved:
            self.events.append(('block_hitl', action))
            return False, 'needs_human_approval'
        self.events.append(('allow', action))
        return True, 'ok'

    def incident(self, name, severity):
        self.events.append(('incident', name, severity))
        return {'containment': 'disable_tool:' + name, 'notify': severity >= 'sev2'}


gov = RuntimeGovernor(hitl_actions={'employment_decision', 'credit_decision'})
print('=== Runtime Governance ===')
print(gov.authorize('summarize_ticket'))
print(gov.authorize('employment_decision'))
print(gov.authorize('employment_decision', user_approved=True))
print(gov.incident('jailbreak_spike', 'sev2'))
print('trail:', gov.events)
print(f'\nKey: Compliance continues after deploy—authorization + incident trails are mandatory evidence.')


## 课后思考题

1. GPAI 与高风险系统的义务如何同时落在“基座模型供应商”和“应用集成方”？
2. 模型卡里哪些字段对审计最关键？哪些常被虚写？
3. 如何在不存储明文用户内容的前提下保留可审计日志？
4. 合规门禁失败时，灰度发布是否允许？需要哪些补偿控制？

---
> 本节涵盖了18.8 AI 治理与合规的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
